# CFP_TimesFM_Forecasts

Generates 1-step-ahead VaR forecasts using Google TimesFM 2.5 (200M). Uses native quantile head for deciles; Student-t tail interpolation for sub-10% VaR levels.

**Paper:** Pele, Lessmann, Hardle (2026)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

DATA_DIR = Path('../cfp_ijf_data')
RET_DIR = DATA_DIR / 'returns'

ASSETS = [
    'SP500','STOXX','GDAXI','CACT','FTSE100','ICLN','NIKKEI','HSI','BOVESPA','NIFTY','ASX200',
    'CBU0','TLT','IBGL','DJCI','GOLD','WTI','NATGAS','BTC','ETH','EURUSD','GBPUSD','USDJPY','AUDUSD'
]
ALPHAS = [0.01, 0.025, 0.05, 0.10]
CONTEXT = 512
N_SAMPLES = 1000

def load_returns(asset):
    df = pd.read_csv(RET_DIR / f'{asset}.csv', parse_dates=['date']).set_index('date').sort_index()
    df = df[df['log_return'].abs() <= 0.50]
    return df['log_return']

print(f'Assets: {len(ASSETS)}, Context: {CONTEXT}, Samples: {N_SAMPLES}')

MODELS = [('timesfm25', 'google/timesfm-2.5-200m-pytorch', 'TimesFM-2.5')]

In [ ]:
import timesfm
from scipy.optimize import minimize
from scipy.stats import t as t_dist

tfm = timesfm.TimesFm(
    hparams=timesfm.TimesFmHparams(per_core_batch_size=32, horizon_len=1, context_len=CONTEXT),
    checkpoint=timesfm.TimesFmCheckpoint(huggingface_repo_id='google/timesfm-2.5-200m-pytorch'),
)

QUANTILE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

def fit_student_t(quantile_vals, quantile_levels):
    def objective(params):
        nu, mu, sigma = params
        if nu <= 2 or sigma <= 0:
            return 1e10
        predicted = [t_dist.ppf(q, df=nu, loc=mu, scale=sigma) for q in quantile_levels]
        return sum((p - v)**2 for p, v in zip(predicted, quantile_vals))
    res = minimize(objective, [5.0, np.median(quantile_vals), np.std(quantile_vals)],
                   method='Nelder-Mead')
    return res.x

out_dir = DATA_DIR / 'timesfm25'
out_dir.mkdir(exist_ok=True)

for asset in tqdm(ASSETS, desc='TimesFM-2.5'):
    ret = load_returns(asset)
    n = len(ret)
    vals = ret.values
    dates = ret.index
    records = []
    for t_idx in range(CONTEXT, n):
        context = vals[t_idx-CONTEXT:t_idx].tolist()
        _, q_out = tfm.forecast_on_df(
            pd.DataFrame({'unique_id': [asset]*CONTEXT, 'ds': range(CONTEXT), 'y': context}),
            freq='D', value_name='y', num_jobs=1,
        )
        # Extract quantile forecasts
        q_vals = [q_out.iloc[0].get(f'timesfm-q-{q}', np.nan) for q in QUANTILE_LEVELS]
        if any(np.isnan(q_vals)):
            records.append({'date': dates[t_idx], **{f'VaR_{a}': np.nan for a in ALPHAS}, 'mean': np.nan, 'std': np.nan})
            continue
        # Fit Student-t for tail extrapolation
        nu, mu, sigma = fit_student_t(q_vals, QUANTILE_LEVELS)
        row = {'date': dates[t_idx]}
        for alpha in ALPHAS:
            if alpha >= 0.10:
                row[f'VaR_{alpha}'] = np.interp(alpha, QUANTILE_LEVELS, q_vals)
            else:
                row[f'VaR_{alpha}'] = t_dist.ppf(alpha, df=max(nu,2.1), loc=mu, scale=max(sigma,1e-8))
        row['mean'] = mu
        row['std'] = sigma
        records.append(row)
    df_out = pd.DataFrame(records).set_index('date')
    df_out.to_parquet(out_dir / f'{asset}.parquet')
print(f'TimesFM-2.5: {len(ASSETS)} assets saved')

In [ ]:
# Verify outputs
from pathlib import Path
for model_slug, _, label in MODELS:
    out_dir = DATA_DIR / model_slug
    files = list(out_dir.glob('*.parquet'))
    print(f'{label}: {len(files)} parquet files')